In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develop a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-10-27
Last Modified: 2026-10-27
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import numpy as np
import matplotlib.pyplot as plt

from sg.models import Encoder, StrategyEncoder
from core.data import default_sessions, subject_ids, session_ids
from core.viz import plot_grouped_boxes
from utils.colors import colors_region
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
do_bweights = True
do_r2 = False

In [ ]:
if do_bweights:
    for subj_id in ["MR82", "MR83", "MR94", "MR95"]:
        encoder = Encoder(
            subj_id,
            default_sessions[subj_id],
            binwidth_ms=100,
            norm=True,
        )
        encoder.verify()

        encoder_mb = StrategyEncoder(
            subj_id,
            default_sessions[subj_id],
            binwidth_ms=100,
            strategy_params="mb",
            add_interaction=True,
            add_svd=True,
            num_svd=20,
            norm=True,
            strategy_filter="mb",
        )
        encoder_mf = StrategyEncoder(
            subj_id,
            default_sessions[subj_id],
            binwidth_ms=100,
            strategy_params="mf",
            add_interaction=True,
            add_svd=True,
            num_svd=20,
            norm=True,
            strategy_filter="mf",
        )
        encoder_mb.verify()
        encoder_mf.verify()

        weights = {
            strat: {reg: [] for reg in encoder.regions} for strat in ["mb", "mf"]
        }
        for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
            e_mb = StrategyEncoder(
                subj_id,
                sess_id,
                binwidth_ms=100,
                strategy_params="mb",
                balance_strategy=False,
                add_interaction=True,
                add_svd=True,
                num_svd=20,
                norm=True,
                strategy_filter="mb",
            )
            e_mf = StrategyEncoder(
                subj_id,
                sess_id,
                binwidth_ms=100,
                strategy_params="mf",
                balance_strategy=False,
                add_interaction=True,
                add_svd=True,
                num_svd=20,
                norm=True,
                strategy_filter="mf",
            )

            try:
                e_mb.fit_encoder()
                e_mf.fit_encoder()
            except ValueError:
                continue

            assert np.all(e_mb.regions == e_mf.regions)
            for reg in e_mb.regions:
                weights["mb"][reg].append(e_mb.encoder_weights[e_mb.reg_idxs[reg]])
                weights["mf"][reg].append(e_mf.encoder_weights[e_mb.reg_idxs[reg]])

        weights = {
            strat: {reg: np.concatenate(weights[strat][reg]) for reg in encoder.regions}
            for strat in ["mb", "mf"]
        }

        models_strat = {"mb": encoder_mb, "mf": encoder_mf}
        tv_idxs = {strat: model.tv_idxs for (strat, model) in models_strat.items()}
        tv_names = {strat: model.tv_names for (strat, model) in models_strat.items()}

        fig, axes = plt.subplots(
            ncols=3, nrows=2, figsize=(8, 7), sharey=True, tight_layout=True
        )

        for i, (strat, weights_strat) in enumerate(weights.items()):
            for j, tv_type in enumerate(["base_regressors", "interaction", "svd"]):
                _, ax = plot_grouped_boxes(
                    data={
                        regr: {
                            reg: np.abs(weights_strat[reg][:, tv_idxs[strat][regr]])
                            for reg in encoder.regions
                        }
                        for regr in tv_names[strat][tv_type]
                    },
                    colors=colors_region,
                    ylim=[-0.1, 1],
                    ax=axes[i][j],
                )

                if i == 0:
                    ax.set_title(tv_type)
                if j == 0:
                    ax.set_ylabel(rf"$\beta$ distribution, {strat}")

        fig.suptitle(subj_id)

        save_fig(
            fig,
            FIGURES_DIR / "bweight" / "boxplot_comp" / "svd_20",
            f"{subj_id}_bweight_comp_rect.png",
        )

In [ ]:
from sg.models import ShuffledEncoder

se = ShuffledEncoder(
    subj_id="MR82",
    sess_id=default_sessions["MR82"],
)
se.get_cvr2(pivot="response")
se.get_cvr2(pivot="rewarded")

In [ ]:
from sg.models import ShuffledEncoder
import pickle
from utils.paths import VARS_DIR

for subj_id in ["MR82", "MR83", "MR94", "MR95"]:
    print(f"> {subj_id}")
    encoder = Encoder(
        subj_id,
        default_sessions[subj_id],
        binwidth_ms=100,
        norm=True,
    )
    encoder.get_data()
    e_mb = StrategyEncoder(
        subj_id,
        default_sessions[subj_id],
        binwidth_ms=100,
        strategy_params="mb",
        balance_strategy=False,
        add_interaction=True,
        norm=True,
        strategy_filter="mb",
    )

    e_mf = StrategyEncoder(
        subj_id,
        default_sessions[subj_id],
        binwidth_ms=100,
        strategy_params="mf",
        balance_strategy=False,
        add_interaction=True,
        norm=True,
        strategy_filter="mf",
    )
    e_mb.build_dm()
    e_mf.build_dm()
    model_strat = {"mb": e_mb, "mf": e_mf}

    cvr2s = {
        strat: {
            reg: {regr: [] for regr in model.tv_idxs.keys()} for reg in encoder.regions
        }
        for (strat, model) in model_strat.items()
    }
    dr2s = {
        strat: {
            reg: {regr: [] for regr in model.tv_idxs.keys()} for reg in encoder.regions
        }
        for (strat, model) in model_strat.items()
    }

    for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
        print(f">>{sess_id}")
        try:
            se_mb = ShuffledEncoder(
                subj_id=subj_id,
                sess_id=sess_id,
                enc_class=StrategyEncoder,
                binwidth_ms=100,
                strategy_params="mb",
                balance_strategy=False,
                add_interaction=True,
                norm=True,
                strategy_filter="mb",
            )
            se_mf = ShuffledEncoder(
                subj_id=subj_id,
                sess_id=sess_id,
                enc_class=StrategyEncoder,
                binwidth_ms=100,
                strategy_params="mf",
                balance_strategy=False,
                add_interaction=True,
                norm=True,
                strategy_filter="mf",
            )

            se_mb.get_cvr2_all(n_iters=2)
            se_mb.get_dr2_all(n_iters=2)

            se_mf.get_cvr2_all(n_iters=2)
            se_mf.get_dr2_all(n_iters=2)
        except ValueError as e:
            print(f"{subj_id} {sess_id} skipped: {e}")
            continue

        assert all(se_mb.encoder_full.regions == se_mf.encoder_full.regions)

        for reg in se_mb.encoder_full.regions:
            for strat, se in {"mb": se_mb, "mf": se_mf}.items():
                for regr in se.regressors:
                    cvr2s[strat][reg][regr].append(
                        se.cvr2_unit[regr].mean(axis=0)[se.encoder_full.reg_idxs[reg]]
                    )
                    dr2s[strat][reg][regr].append(
                        se.dr2_unit[regr].mean(axis=0)[se.encoder_full.reg_idxs[reg]]
                    )

    fpath = VARS_DIR / "cv_d_r2"
    fpath.mkdir(exist_ok=True)
    with open(fpath / f"{subj_id}_cvr2s.pkl", "wb") as f:
        pickle.dump(cvr2s, f)
    with open(fpath / f"{subj_id}_dr2s.pkl", "wb") as f:
        pickle.dump(dr2s, f)

    cvr2s = {
        strat: {
            reg: {
                regr: np.concatenate(cvr2s[strat][reg][regr])
                for regr in cvr2s[strat][reg].keys()
            }
            for reg in encoder.regions
        }
        for strat in ["mb", "mf"]
    }

    dr2s = {
        strat: {
            reg: {
                regr: np.concatenate(dr2s[strat][reg][regr])
                for regr in dr2s[strat][reg].keys()
            }
            for reg in encoder.regions
        }
        for strat in ["mb", "mf"]
    }

    models_strat = {"mb": e_mb, "mf": e_mf}
    tv_names = {strat: model.tv_names for (strat, model) in models_strat.items()}

    fig, axes = plt.subplots(
        ncols=2, nrows=2, figsize=(8, 7), sharey=True, tight_layout=True
    )

    for i, (strat, cvr2_strat) in enumerate(cvr2s.items()):
        for j, tv_type in enumerate(["base_regressors", "interaction"]):
            _, ax = plot_grouped_boxes(
                data={
                    regr: {reg: cvr2_strat[reg][regr] for reg in encoder.regions}
                    for regr in tv_names[strat][tv_type]
                },
                colors=colors_region,
                ylim=[-0.1, 0.8],
                ax=axes[i][j],
            )

            if i == 0:
                ax.set_title(tv_type)
            if j == 0:
                ax.set_ylabel(rf"cv $r^2$ distribution, {strat}")

    fig.suptitle(subj_id)

    save_fig(
        fig,
        FIGURES_DIR / "cv_d_r2" / "boxplot_comp",
        f"{subj_id}_cvr2_comp.png",
    )

    fig, axes = plt.subplots(
        ncols=2, nrows=2, figsize=(8, 7), sharey=True, tight_layout=True
    )

    for i, (strat, dr2_strat) in enumerate(dr2s.items()):
        for j, tv_type in enumerate(["base_regressors", "interaction"]):
            _, ax = plot_grouped_boxes(
                data={
                    regr: {reg: dr2_strat[reg][regr] for reg in encoder.regions}
                    for regr in tv_names[strat][tv_type]
                },
                colors=colors_region,
                ylim=[-0.1, 0.3],
                ax=axes[i][j],
            )

            if i == 0:
                ax.set_title(tv_type)
            if j == 0:
                ax.set_ylabel(rf"$\Delta$ $r^2$ distribution, {strat}")

    fig.suptitle(subj_id)

    save_fig(
        fig,
        FIGURES_DIR / "cv_d_r2" / "boxplot_comp",
        f"{subj_id}_dr2_comp.png",
    )

In [ ]:
cvr2s = {
    strat: {
        reg: {
            regr: np.concatenate(cvr2s[strat][reg][regr])
            for regr in cvr2s[strat][reg].keys()
            if len(cvr2s[strat][reg][regr]) > 0
        }
        for reg in encoder.regions
    }
    for strat in ["mb", "mf"]
}

dr2s = {
    strat: {
        reg: {
            regr: np.concatenate(dr2s[strat][reg][regr])
            for regr in dr2s[strat][reg].keys()
            if len(dr2s[strat][reg][regr]) > 0
        }
        for reg in encoder.regions
    }
    for strat in ["mb", "mf"]
}

In [ ]:
# also, save cvr2 and dr2 before concatenating

In [ ]:
models_strat = {"mb": e_mb, "mf": e_mf}
tv_names = {strat: model.tv_names for (strat, model) in models_strat.items()}

fig, axes = plt.subplots(
    ncols=3, nrows=2, figsize=(8, 7), sharey=True, tight_layout=True
)

for i, (strat, cvr2_strat) in enumerate(cvr2s.items()):
    for j, tv_type in enumerate(["task_vars", "strategy_params"]):
        _, ax = plot_grouped_boxes(
            data={
                regr: {reg: cvr2_strat[reg][regr] for reg in encoder.regions}
                for regr in tv_names[strat][tv_type]
            },
            colors=colors_region,
            ylim=[-0.1, 1],
            ax=axes[i][j],
        )

        if i == 0:
            ax.set_title(tv_type)
        if j == 0:
            ax.set_ylabel(rf"cv $r^2$ distribution, {strat}")

fig.suptitle(subj_id)

save_fig(
    fig,
    FIGURES_DIR / "cv_d_r2" / "boxplot_comp",
    f"{subj_id}_cvr2_comp.png",
)

fig, axes = plt.subplots(
    ncols=3, nrows=2, figsize=(8, 7), sharey=True, tight_layout=True
)

for i, (strat, dr2_strat) in enumerate(dr2s.items()):
    for j, tv_type in enumerate(["base_regressors", "interactions"]):
        _, ax = plot_grouped_boxes(
            data={
                regr: {reg: dr2_strat[reg][regr] for reg in encoder.regions}
                for regr in tv_names[strat][tv_type]
            },
            colors=colors_region,
            ylim=[-0.1, 1],
            ax=axes[i][j],
        )

        if i == 0:
            ax.set_title(tv_type)
        if j == 0:
            ax.set_ylabel(rf"$\Delta$ $r^2$ distribution, {strat}")

fig.suptitle(subj_id)

save_fig(
    fig,
    FIGURES_DIR / "cv_d_r2" / "boxplot_comp",
    f"{subj_id}_dr2_comp.png",
)

In [ ]:
e_mb = StrategyEncoder(
    subj_id,
    default_sessions[subj_id],
    binwidth_ms=100,
    strategy_params="mb",
    balance_strategy=False,
    norm=True,
    strategy_filter="mb",
)

e_mf = StrategyEncoder(
    subj_id,
    default_sessions[subj_id],
    binwidth_ms=100,
    strategy_params="mf",
    balance_strategy=False,
    norm=True,
    strategy_filter="mf",
)
e_mb.build_dm()
e_mf.build_dm()